# Notebook 03: Building the Training Dataset

I loop the feature function over every trail and combine the results with the difficulty labels into one table: **one row per trail**. That table is what the model will eventually learn from.

**New idea here:** instead of redefining the functions, I will *import* them from `src/features.py`. That file is now the single source of truth.

In [1]:
import sys
sys.path.append('../src')   # let the notebook see our src/ folder

import pandas as pd
from features import load_trail, compute_features

## 1. Load the labels

Each row pairs a GPX filename with its Wikiloc difficulty, the ground truth.

In [2]:
labels = pd.read_csv('../data/labels.csv')
print(labels.shape)
labels['difficulty'].value_counts()

(116, 2)


difficulty
moderate          29
easy              29
difficult         29
very difficult    29
Name: count, dtype: int64

## 2. Loop over every trail

For each file: I load it, compute its features, and attach its label. I wrap it in `try/except` so one bad file won't crash the whole run.

In [3]:
rows = []
for _, r in labels.iterrows():
    try:
        df = load_trail('../data/raw/gpx/' + r['filename'])
        feats = compute_features(df)
        feats['difficulty'] = r['difficulty']
        feats['filename'] = r['filename']
        rows.append(feats)
    except Exception as e:
        print('FAILED:', r['filename'], '->', e)

data = pd.DataFrame(rows)
print('Feature table:', data.shape)
data.head()

Feature table: (116, 14)


,distance_km,elevation_gain_m,elevation_loss_m,gain_rate_m_per_km,max_elevation_m,min_elevation_m,elevation_range_m,mean_slope_deg,max_slope_deg,slope_variability,pct_steep_segments,exposure_index,difficulty,filename
0,13.48,1301,1301,96.5,1847,792,1055,13.4,61.0,9.9,18.8,0.22,moderate,dolomitas-anello-della-cima-dellalbero-gruppo-...
1,5.95,243,284,40.8,186,38,149,7.8,61.0,10.4,8.6,0.22,easy,cerdena-cala-fuili-cala-luna-vuelta-en-barco.gpx
2,9.38,568,568,60.6,1085,693,392,8.6,47.5,8.4,10.3,0.19,difficult,steamers-2021-update.gpx
3,12.21,606,621,49.6,1645,1271,374,7.8,50.6,7.8,7.7,0.12,moderate,mount-baker-wa-usa-chain-lakes-trail.gpx
4,8.09,967,963,119.6,2570,1615,955,17.8,60.5,11.7,35.2,0.15,very difficult,anello-monte-camicia-monte-tramoggia-campo-imp...


## 3. Do the features actually relate to difficulty?

Before any modelling, I eyeball the averages per difficulty level. If terrain encodes difficulty, harder trails should show bigger numbers for climbing and steepness.

Notice in the table below, how *very difficult* trails don't always have the biggest distance. Some are short but brutally steep, others (like the multi-day Amatola legs) are long but graded. That nuance is exactly what the model will learn to weigh.

In [4]:
order = ['easy', 'moderate', 'difficult', 'very difficult']
summary = (data.groupby('difficulty')[['distance_km','elevation_gain_m','gain_rate_m_per_km','mean_slope_deg','pct_steep_segments']]
               .mean().reindex(order).round(1))
summary

,distance_km,elevation_gain_m,gain_rate_m_per_km,mean_slope_deg,pct_steep_segments
difficulty,,,,,
easy,9.6,415.9,45.3,7.1,7.2
moderate,15.1,838.4,59.3,8.9,11.1
difficult,12.5,911.2,72.2,10.3,14.5
very difficult,23.3,1361.7,73.1,10.7,14.9


### What this tells us

Across all 116 trails (a balanced 29 per level), the terrain features now rise cleanly with difficulty, and the small-sample wobble from the 12-trail pilot has smoothed out. Steepness (`mean_slope_deg`), total climb, and climb-rate all increase step by step from easy to very difficult. **Steepness clearly tracks difficulty, and the core thesis holds.**

The sharper insight is at the top end: `difficult` and `very difficult` have almost identical steepness (10.3 deg vs 10.7 deg). What separates them is **distance (12.5 -> 23.3 km) and total climb (911 -> 1362 m)**.

**Takeaway: difficulty is multi-dimensional.** Some trails are hard because they are steep; others because they are long. No single feature captures it, which is exactly why we need a *model* that combines features rather than a simple threshold rule. It is also what SHAP will later reveal per trail: "this one is hard because of distance; that one because of slope."

## 4. Save the dataset

Write the table to `data/processed/features.csv` so the modelling notebook can pick it up later.

In [5]:
import os
os.makedirs('../data/processed', exist_ok=True)
data.to_csv('../data/processed/features.csv', index=False)
print('Saved', len(data), 'trails to data/processed/features.csv')

Saved 116 trails to data/processed/features.csv


## 5. What's next

You now have a balanced **116-trail dataset** (29 per difficulty level) — plenty to train a first model. Next:

1. Train XGBoost vs Random Forest
2. Use SHAP to see which features drive difficulty and confirm the endurance-vs-steepness story.